# Подготовка обучающей и тестовой выборки, кросс-валидация и подбор гиперпараметров на примере метода ближайших соседей.

In [114]:
import polars as pl

data = pl.read_csv("Titanic-Dataset.csv").drop("Ticket", "Name")
data

PassengerId,Survived,Pclass,Sex,Age,SibSp,Parch,Fare,Cabin,Embarked
i64,i64,i64,str,f64,i64,i64,f64,str,str
1,0,3,"""male""",22.0,1,0,7.25,null,"""S"""
2,1,1,"""female""",38.0,1,0,71.2833,"""C85""","""C"""
3,1,3,"""female""",26.0,0,0,7.925,null,"""S"""
4,1,1,"""female""",35.0,1,0,53.1,"""C123""","""S"""
5,0,3,"""male""",35.0,0,0,8.05,null,"""S"""
…,…,…,…,…,…,…,…,…,…
887,0,2,"""male""",27.0,0,0,13.0,null,"""S"""
888,1,1,"""female""",19.0,0,0,30.0,"""B42""","""S"""
889,0,3,"""female""",null,1,2,23.45,null,"""S"""


`Cabin` имеет 687 пропусков, лучше данный признак удалить

In [115]:
data = data.drop("Cabin")
data

PassengerId,Survived,Pclass,Sex,Age,SibSp,Parch,Fare,Embarked
i64,i64,i64,str,f64,i64,i64,f64,str
1,0,3,"""male""",22.0,1,0,7.25,"""S"""
2,1,1,"""female""",38.0,1,0,71.2833,"""C"""
3,1,3,"""female""",26.0,0,0,7.925,"""S"""
4,1,1,"""female""",35.0,1,0,53.1,"""S"""
5,0,3,"""male""",35.0,0,0,8.05,"""S"""
…,…,…,…,…,…,…,…,…
887,0,2,"""male""",27.0,0,0,13.0,"""S"""
888,1,1,"""female""",19.0,0,0,30.0,"""S"""
889,0,3,"""female""",null,1,2,23.45,"""S"""


In [116]:
from sklearn.preprocessing import LabelEncoder

sex_le = LabelEncoder()

data = data.with_columns(
    Sex=sex_le.fit_transform(data.get_column("Sex")),
)
data

PassengerId,Survived,Pclass,Sex,Age,SibSp,Parch,Fare,Embarked
i64,i64,i64,i64,f64,i64,i64,f64,str
1,0,3,1,22.0,1,0,7.25,"""S"""
2,1,1,0,38.0,1,0,71.2833,"""C"""
3,1,3,0,26.0,0,0,7.925,"""S"""
4,1,1,0,35.0,1,0,53.1,"""S"""
5,0,3,1,35.0,0,0,8.05,"""S"""
…,…,…,…,…,…,…,…,…
887,0,2,1,27.0,0,0,13.0,"""S"""
888,1,1,0,19.0,0,0,30.0,"""S"""
889,0,3,0,null,1,2,23.45,"""S"""


In [117]:
data = data.hstack(data.get_column("Embarked").to_dummies()).drop("Embarked")
data

PassengerId,Survived,Pclass,Sex,Age,SibSp,Parch,Fare,Embarked_C,Embarked_Q,Embarked_S,Embarked_null
i64,i64,i64,i64,f64,i64,i64,f64,u8,u8,u8,u8
1,0,3,1,22.0,1,0,7.25,0,0,1,0
2,1,1,0,38.0,1,0,71.2833,1,0,0,0
3,1,3,0,26.0,0,0,7.925,0,0,1,0
4,1,1,0,35.0,1,0,53.1,0,0,1,0
5,0,3,1,35.0,0,0,8.05,0,0,1,0
…,…,…,…,…,…,…,…,…,…,…,…
887,0,2,1,27.0,0,0,13.0,0,0,1,0
888,1,1,0,19.0,0,0,30.0,0,0,1,0
889,0,3,0,null,1,2,23.45,0,0,1,0


In [118]:
data = data.with_columns(
    Age=pl.col("Age").fill_null(pl.col("Age").median().over("Pclass", "Sex"))
)
data

PassengerId,Survived,Pclass,Sex,Age,SibSp,Parch,Fare,Embarked_C,Embarked_Q,Embarked_S,Embarked_null
i64,i64,i64,i64,f64,i64,i64,f64,u8,u8,u8,u8
1,0,3,1,22.0,1,0,7.25,0,0,1,0
2,1,1,0,38.0,1,0,71.2833,1,0,0,0
3,1,3,0,26.0,0,0,7.925,0,0,1,0
4,1,1,0,35.0,1,0,53.1,0,0,1,0
5,0,3,1,35.0,0,0,8.05,0,0,1,0
…,…,…,…,…,…,…,…,…,…,…,…
887,0,2,1,27.0,0,0,13.0,0,0,1,0
888,1,1,0,19.0,0,0,30.0,0,0,1,0
889,0,3,0,21.5,1,2,23.45,0,0,1,0


In [119]:
from sklearn.preprocessing import MinMaxScaler

pclass_scaler = MinMaxScaler()
age_scaler = MinMaxScaler()
sibsp_scaler = MinMaxScaler()
parch_scaler = MinMaxScaler()
fare_scaler = MinMaxScaler()

data = data.with_columns(
    Pclass=pl.Series(
        pclass_scaler.fit_transform(data.get_column("Pclass").reshape((-1, 1)))
    ).list.explode(),
    Age=pl.Series(
        age_scaler.fit_transform(data.get_column("Age").reshape((-1, 1)))
    ).list.explode(),
    SibSp=pl.Series(
        sibsp_scaler.fit_transform(data.get_column("SibSp").reshape((-1, 1)))
    ).list.explode(),
    Parch=pl.Series(
        parch_scaler.fit_transform(data.get_column("Parch").reshape((-1, 1)))
    ).list.explode(),
    Fare=pl.Series(
        fare_scaler.fit_transform(data.get_column("Fare").reshape((-1, 1)))
    ).list.explode(),
)
data

PassengerId,Survived,Pclass,Sex,Age,SibSp,Parch,Fare,Embarked_C,Embarked_Q,Embarked_S,Embarked_null
i64,i64,f64,i64,f64,f64,f64,f64,u8,u8,u8,u8
1,0,1.0,1,0.271174,0.125,0.0,0.014151,0,0,1,0
2,1,0.0,0,0.472229,0.125,0.0,0.139136,1,0,0,0
3,1,1.0,0,0.321438,0.0,0.0,0.015469,0,0,1,0
4,1,0.0,0,0.434531,0.125,0.0,0.103644,0,0,1,0
5,0,1.0,1,0.434531,0.0,0.0,0.015713,0,0,1,0
…,…,…,…,…,…,…,…,…,…,…,…
887,0,0.5,1,0.334004,0.0,0.0,0.025374,0,0,1,0
888,1,0.0,0,0.233476,0.0,0.0,0.058556,0,0,1,0
889,0,1.0,0,0.264891,0.125,0.333333,0.045771,0,0,1,0


In [120]:
data = data.drop("PassengerId")
data

Survived,Pclass,Sex,Age,SibSp,Parch,Fare,Embarked_C,Embarked_Q,Embarked_S,Embarked_null
i64,f64,i64,f64,f64,f64,f64,u8,u8,u8,u8
0,1.0,1,0.271174,0.125,0.0,0.014151,0,0,1,0
1,0.0,0,0.472229,0.125,0.0,0.139136,1,0,0,0
1,1.0,0,0.321438,0.0,0.0,0.015469,0,0,1,0
1,0.0,0,0.434531,0.125,0.0,0.103644,0,0,1,0
0,1.0,1,0.434531,0.0,0.0,0.015713,0,0,1,0
…,…,…,…,…,…,…,…,…,…,…
0,0.5,1,0.334004,0.0,0.0,0.025374,0,0,1,0
1,0.0,0,0.233476,0.0,0.0,0.058556,0,0,1,0
0,1.0,0,0.264891,0.125,0.333333,0.045771,0,0,1,0


In [121]:
y = data.get_column("Survived")
X = data.drop("Survived")

In [122]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y,
)

In [123]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import classification_report

knn_base = KNeighborsClassifier(n_neighbors=5)

knn_base.fit(X_train, y_train)
y_pred_base = knn_base.predict(X_test)

print("Base KNN (k=5)")
print(classification_report(y_test, y_pred_base))

Base KNN (k=5)
              precision    recall  f1-score   support

           0       0.80      0.88      0.84       110
           1       0.78      0.65      0.71        69

    accuracy                           0.79       179
   macro avg       0.79      0.77      0.77       179
weighted avg       0.79      0.79      0.79       179



In [124]:
from sklearn.model_selection import GridSearchCV, KFold, StratifiedKFold

param_grid = {
    "n_neighbors": list(range(1, 26)),
    "weights": ["uniform", "distance"],
    "metric": ["euclidean", "manhattan"],
}

cv_strategy_1 = KFold(
    n_splits=5,
    shuffle=True,
    random_state=42,
)

cv_strategy_2 = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42,
)

for cv in (cv_strategy_1, cv_strategy_2):
    grid_search = GridSearchCV(
        estimator=KNeighborsClassifier(),
        param_grid=param_grid,
        scoring="f1",
        cv=cv,
    )

    grid_search.fit(X_train, y_train)

    print(cv)
    print("Best params (GridSearchCV):", grid_search.best_params_)
    y_pred_base = grid_search.predict(X_test)
    print(classification_report(y_test, y_pred_base))

KFold(n_splits=5, random_state=42, shuffle=True)
Best params (GridSearchCV): {'metric': 'manhattan', 'n_neighbors': 9, 'weights': 'uniform'}
              precision    recall  f1-score   support

           0       0.80      0.90      0.85       110
           1       0.80      0.64      0.71        69

    accuracy                           0.80       179
   macro avg       0.80      0.77      0.78       179
weighted avg       0.80      0.80      0.79       179

StratifiedKFold(n_splits=5, random_state=42, shuffle=True)
Best params (GridSearchCV): {'metric': 'euclidean', 'n_neighbors': 3, 'weights': 'uniform'}
              precision    recall  f1-score   support

           0       0.82      0.88      0.85       110
           1       0.78      0.68      0.73        69

    accuracy                           0.80       179
   macro avg       0.80      0.78      0.79       179
weighted avg       0.80      0.80      0.80       179



In [125]:
from sklearn.model_selection import RandomizedSearchCV
from scipy.stats import randint

param_distributions = {
    "n_neighbors": randint(1, 50),
    "weights": ["uniform", "distance"],
    "metric": ["euclidean", "manhattan"],
}

for cv in (cv_strategy_1, cv_strategy_2):
    random_search = RandomizedSearchCV(
        estimator=KNeighborsClassifier(),
        param_distributions=param_distributions,
        n_iter=30,
        scoring="f1",
        cv=cv,
        random_state=42,
    )

    random_search.fit(X_train, y_train)

    print(cv)
    print("Best params (RandomizedSearchCV):", random_search.best_params_)
    y_pred_base = random_search.predict(X_test)
    print(classification_report(y_test, y_pred_base))

KFold(n_splits=5, random_state=42, shuffle=True)
Best params (RandomizedSearchCV): {'metric': 'euclidean', 'n_neighbors': 7, 'weights': 'uniform'}
              precision    recall  f1-score   support

           0       0.79      0.91      0.84       110
           1       0.81      0.61      0.69        69

    accuracy                           0.79       179
   macro avg       0.80      0.76      0.77       179
weighted avg       0.80      0.79      0.79       179

StratifiedKFold(n_splits=5, random_state=42, shuffle=True)
Best params (RandomizedSearchCV): {'metric': 'manhattan', 'n_neighbors': 6, 'weights': 'distance'}
              precision    recall  f1-score   support

           0       0.84      0.88      0.86       110
           1       0.79      0.72      0.76        69

    accuracy                           0.82       179
   macro avg       0.81      0.80      0.81       179
weighted avg       0.82      0.82      0.82       179



|Модель|Accuracy|F1(для выживших)|
|:-:|:-:|:-:|
|База (k=5, euclidean, uniform)|0.79|0.71|
|Лучший Grid + StratifiedKFold|0.80|0.73|
|Лучший Randomized + StratifiedKFold|**0.82**|**0.76**|

Подбор гиперпараметров и выбор стратегии кросс‑валидации дали +3 процентных пункта к accuracy и заметный прирост F1 и recall по классу 1 (выжившие).

Наиболее устойчивые и качественные результаты показала комбинация RandomizedSearchCV + StratifiedKFold.